<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/10_Avalia%C3%A7%C3%A3o%20Integrada%20do%20M%C3%B3dulo%201%20%E2%80%94%20SCADA-Core%20Seguran%C3%A7a%20%26%20Diagn%C3%B3stico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-01-logica/10_Avaliacao_Motor_Integrado_SCADA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 10 - Avaliação do Módulo 1: Motor Integrado de Intertravamento e Diagnóstico

## Sistema SCADA aplicado a um Drone Agrícola de Pulverização — Grupo 06

Este notebook consolida o trabalho desenvolvido ao longo do **Módulo 1 (Lógica Formal & Sistemas Especialistas)**, unindo dois motores construídos separadamente em aulas anteriores:

1. **Motor de Intertravamento** (Aulas 03, 04 e 05) — lógica proposicional estática que define os *permissivos* de decolagem e de pulverização, com prova formal de que estados de perigo são logicamente impossíveis sob a regra de intertravamento.
2. **Motor de Diagnóstico / Sistema Especialista** (Aulas 07, 08 e 09) — base de regras `SE...ENTÃO` avaliada por **Forward Chaining** (diagnóstico contínuo a partir dos sensores) e **Backward Chaining** (verificação de hipóteses específicas), capaz de identificar causas raiz como obstrução de bicos ou vazamento.

O objetivo desta avaliação é demonstrar que **nenhum dos dois motores isoladamente é suficiente** para garantir a segurança da planta: o intertravamento reage a limites estáticos (bateria, vento, GPS, tanque vazio), enquanto o sistema especialista identifica **padrões combinados** entre variáveis (ex: bomba ligada + vazão baixa + pressão alta) que só fazem sentido quando avaliados em conjunto. A união dos dois motores é o que efetivamente impede a planta de entrar em uma combinação operacional de risco, cumprindo o objetivo geral do projeto SCADA-Core.

### Recapitulação dos entregáveis do Módulo 1

| Aula | Tema | Entregável |
|---|---|---|
| 02 | Representação Simbólica | Catálogo de Tags (ISA 5.1) e proposições atômicas |
| 03 | Tautologias e Contradições | Prova formal do intertravamento de emergência |
| 04 | Lógica Proposicional: Conectivos | Blocos de permissivo (decolagem / pulverização) |
| 05 | Formas Normais | Otimização booleana das expressões de intertravamento |
| 07 | Validade e Inferência Lógica | Regras de inferência aplicadas ao diagnóstico de falhas |
| 09 | Motor de Inferência | Forward/Backward Chaining + base de regras de diagnóstico |
| **10** | **Avaliação do Módulo 1** | **Motor integrado de Intertravamento e Diagnóstico (este notebook)** |


---
## 1. Motor de Intertravamento (recapitulando as Aulas 03–05)

Os blocos abaixo reconstroem, a partir dos notebooks anteriores, os operadores lógicos fundamentais e os dois permissivos de segurança do drone: **decolagem** e **pulverização**.


In [1]:
from typing import Dict, List, Set, Tuple
import itertools
import pandas as pd

# Operadores fundamentais da lógica proposicional (Aula 04)
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    return p == q

print("Operadores lógicos proposicionais carregados com sucesso.")


Operadores lógicos proposicionais carregados com sucesso.


In [2]:
# Permissivo de decolagem (Aula 04)
def permissivo_decolagem_drone(gps_ok: bool, bat_low: bool, wind_high: bool, e_stop: bool,
                                auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    modo_valido = XOR(auto_mode, manual_mode)

    permissivo = (gps_ok and
                  NOT(bat_low) and
                  NOT(wind_high) and
                  NOT(e_stop) and
                  modo_valido)

    trip = NOT(gps_ok) or bat_low or wind_high or e_stop

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }


# Permissivo de pulverização (Aula 04)
def permissivo_pulverizacao(is_flying: bool, alt_ok: bool, tank_empty: bool) -> Dict[str, bool]:
    permissivo = is_flying and alt_ok and NOT(tank_empty)
    trip = NOT(is_flying) or NOT(alt_ok) or tank_empty

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip
    }

print("Blocos de permissivo (decolagem e pulverização) carregados.")


Blocos de permissivo (decolagem e pulverização) carregados.


### Prova de contradição do estado de perigo (Aula 03)

Reaproveita-se aqui a demonstração de que, sob a regra de intertravamento `e_stop -> ¬motores_armados`, o estado de perigo `e_stop AND motores_armados` é uma **contradição lógica** — ou seja, jamais poderá ocorrer se a regra for respeitada pela controladora de voo.


In [3]:
def prova_estado_perigo() -> pd.DataFrame:
    """Verifica exaustivamente que S_perigo = (e1 AND m1) é sempre FALSO
    quando a regra de intertravamento (e1 -> not m1) está ativa."""
    linhas = []
    for e1, m1 in itertools.product([False, True], repeat=2):
        s_perigo = e1 and m1
        regra_interlock = (not e1) or (not m1)
        phi = s_perigo and regra_interlock
        linhas.append([e1, m1, s_perigo, regra_interlock, phi])

    df = pd.DataFrame(linhas, columns=[
        'Emergencia (e1)', 'Motor_Armado (m1)',
        'Estado de Perigo', 'Regra Ativa', 'Sistema em Falha (Phi)'
    ])
    return df

df_prova = prova_estado_perigo()
display(df_prova)

if df_prova['Sistema em Falha (Phi)'].any():
    print("ALERTA: contradição não confirmada — revisar a regra de intertravamento.")
else:
    print("CONFIRMADO: Phi é uma contradição formal. O estado de perigo é logicamente impossível.")


,Emergencia (e1),Motor_Armado (m1),Estado de Perigo,Regra Ativa,Sistema em Falha (Phi)
0,False,False,False,True,False
1,False,True,False,True,False
2,True,False,False,True,False
3,True,True,True,False,False


CONFIRMADO: Phi é uma contradição formal. O estado de perigo é logicamente impossível.


---
## 2. Motor de Diagnóstico / Sistema Especialista (recapitulando as Aulas 07–09)

Abaixo estão a base de regras `SE...ENTÃO` e os algoritmos de **Forward Chaining** e **Backward Chaining** implementados na Aula 09, responsáveis por transformar fatos derivados dos sensores em diagnósticos de falha.


In [4]:
# Base de regras de diagnóstico (Aulas 08 e 09)
regras = [
    {"nome": "R1", "condicoes": {"nivel_critico"}, "conclusao": "falta_insumo",
     "descricao": "Nível crítico indica falta de insumo."},
    {"nome": "R2", "condicoes": {"bomba_ligada", "valvula_aberta", "vazao_baixa"},
     "conclusao": "falha_pulverizacao",
     "descricao": "Bomba e válvula ativas com vazão baixa indicam falha de pulverização."},
    {"nome": "R3", "condicoes": {"bomba_ligada", "vazao_baixa", "pressao_alta"},
     "conclusao": "possivel_obstrucao",
     "descricao": "Vazão baixa com pressão alta pode indicar obstrução."},
    {"nome": "R4", "condicoes": {"possivel_obstrucao"}, "conclusao": "interromper_pulverizacao",
     "descricao": "Uma possível obstrução exige interrupção da pulverização."},
    {"nome": "R5", "condicoes": {"falta_insumo"}, "conclusao": "interromper_pulverizacao",
     "descricao": "Falta de insumo exige interrupção da pulverização."},
    {"nome": "R6", "condicoes": {"bateria_critica"}, "conclusao": "interromper_missao",
     "descricao": "Bateria crítica exige interrupção da missão."},
    {"nome": "R7", "condicoes": {"vento_alto"}, "conclusao": "suspender_pulverizacao",
     "descricao": "Vento alto torna a pulverização inadequada."},
    {"nome": "R8", "condicoes": {"bomba_ligada", "vazao_baixa", "pressao_baixa"},
     "conclusao": "possivel_vazamento_ou_falha_bomba",
     "descricao": "Vazão e pressão baixas com a bomba ligada podem indicar vazamento ou falha da bomba."},
]

mensagens_diagnostico = {
    "falta_insumo": "Nível crítico de calda. Realizar abastecimento.",
    "falha_pulverizacao": "Falha no sistema de pulverização.",
    "possivel_obstrucao": "Possível obstrução nos bicos ou na tubulação.",
    "possivel_vazamento_ou_falha_bomba": "Possível vazamento ou falha da bomba.",
    "interromper_pulverizacao": "Interromper a pulverização.",
    "interromper_missao": "Interromper a missão e retornar à base.",
    "suspender_pulverizacao": "Suspender a pulverização devido às condições ambientais.",
}

for regra in regras:
    print(f'{regra["nome"]}: SE {" E ".join(sorted(regra["condicoes"]))} ENTÃO {regra["conclusao"]}')


R1: SE nivel_critico ENTÃO falta_insumo
R2: SE bomba_ligada E valvula_aberta E vazao_baixa ENTÃO falha_pulverizacao
R3: SE bomba_ligada E pressao_alta E vazao_baixa ENTÃO possivel_obstrucao
R4: SE possivel_obstrucao ENTÃO interromper_pulverizacao
R5: SE falta_insumo ENTÃO interromper_pulverizacao
R6: SE bateria_critica ENTÃO interromper_missao
R7: SE vento_alto ENTÃO suspender_pulverizacao
R8: SE bomba_ligada E pressao_baixa E vazao_baixa ENTÃO possivel_vazamento_ou_falha_bomba


In [5]:
def forward_chaining(fatos_iniciais: Set[str], regras: List[dict]) -> Tuple[Set[str], List[dict]]:
    fatos = set(fatos_iniciais)
    trilha = []
    houve_alteracao = True

    while houve_alteracao:
        houve_alteracao = False
        for regra in regras:
            condicoes = regra["condicoes"]
            conclusao = regra["conclusao"]
            if condicoes.issubset(fatos) and conclusao not in fatos:
                fatos.add(conclusao)
                trilha.append({
                    "regra": regra["nome"],
                    "condicoes": sorted(condicoes),
                    "conclusao": conclusao
                })
                houve_alteracao = True

    return fatos, trilha


def backward_chaining(objetivo: str, fatos: Set[str], regras: List[dict],
                       visitados=None, profundidade: int = 0) -> Tuple[bool, List[str]]:
    if visitados is None:
        visitados = set()

    trilha = []
    indentacao = "  " * profundidade

    if objetivo in fatos:
        trilha.append(f"{indentacao}Fato conhecido: {objetivo}")
        return True, trilha

    if objetivo in visitados:
        trilha.append(f"{indentacao}Objetivo já visitado: {objetivo}")
        return False, trilha

    visitados = visitados | {objetivo}
    regras_objetivo = [r for r in regras if r["conclusao"] == objetivo]

    if not regras_objetivo:
        trilha.append(f"{indentacao}Nenhuma regra conclui: {objetivo}")
        return False, trilha

    for regra in regras_objetivo:
        trilha.append(f'{indentacao}Testando {regra["nome"]} para concluir {objetivo}')
        todas_satisfeitas = True

        for condicao in regra["condicoes"]:
            resultado, subtrilha = backward_chaining(condicao, fatos, regras, visitados, profundidade + 1)
            trilha.extend(subtrilha)
            if not resultado:
                todas_satisfeitas = False
                break

        if todas_satisfeitas:
            trilha.append(f'{indentacao}{regra["nome"]} satisfeita -> {objetivo}')
            return True, trilha

    trilha.append(f"{indentacao}Não foi possível comprovar: {objetivo}")
    return False, trilha

print("Motor de inferência (Forward/Backward Chaining) carregado.")


Motor de inferência (Forward/Backward Chaining) carregado.


In [6]:
def medicoes_para_fatos(medicoes: dict) -> Set[str]:
    fatos = set()

    nivel = medicoes["nivel_reservatorio"]
    vazao = medicoes["vazao"]
    pressao = medicoes["pressao"]
    bateria = medicoes["bateria"]
    vento = medicoes["vento"]

    if medicoes.get("bomba_ligada", False):
        fatos.add("bomba_ligada")
    if medicoes.get("valvula_aberta", False):
        fatos.add("valvula_aberta")

    if nivel < 20:
        fatos.add("nivel_baixo")
    if nivel < 5:
        fatos.add("nivel_critico")

    if vazao < 0.5 and medicoes.get("bomba_ligada", False):
        fatos.add("vazao_baixa")
    else:
        fatos.add("vazao_normal")

    if pressao > 5:
        fatos.add("pressao_alta")
    elif pressao < 1 and medicoes.get("bomba_ligada", False):
        fatos.add("pressao_baixa")
    else:
        fatos.add("pressao_normal")

    if bateria < 20:
        fatos.add("bateria_baixa")
    if bateria < 10:
        fatos.add("bateria_critica")

    if vento > 8:
        fatos.add("vento_alto")
    else:
        fatos.add("vento_adequado")

    return fatos


def motor_diagnostico(medicoes: dict, regras: List[dict]) -> dict:
    fatos_iniciais = medicoes_para_fatos(medicoes)
    fatos_finais, trilha = forward_chaining(fatos_iniciais, regras)

    diagnosticos = [
        {"codigo": fato, "mensagem": mensagens_diagnostico[fato]}
        for fato in fatos_finais
        if fato in mensagens_diagnostico
    ]

    return {
        "medicoes": medicoes,
        "fatos_iniciais": sorted(fatos_iniciais),
        "fatos_finais": sorted(fatos_finais),
        "diagnosticos": diagnosticos,
        "regras_ativadas": trilha,
    }

print("Motor de diagnóstico integrado (medições -> fatos -> diagnósticos) carregado.")


Motor de diagnóstico integrado (medições -> fatos -> diagnósticos) carregado.


---
## 3. Integração: Motor SCADA Unificado

Aqui está o núcleo desta avaliação. A função `motor_scada_integrado` recebe a telemetria completa do drone e da estação de solo e:

1. Calcula os **permissivos de intertravamento** (decolagem e pulverização), derivando as booleanas de limiar a partir dos mesmos limites definidos na Aula 02 (`bateria < 20% -> bat_low`, `vento > 8 m/s -> wind_high`, `nível ≤ 3% -> tanque vazio`);
2. Executa o **motor de diagnóstico** (Forward Chaining) sobre a mesma telemetria;
3. **Cruza os dois resultados**: mesmo que o intertravamento estático libere a pulverização (bateria, vento e tanque OK), o comando final da bomba só é liberado se o sistema especialista **não** tiver identificado nenhuma condição crítica (obstrução, vazamento, falta de insumo, falha de pulverização).

Essa etapa evidencia exatamente por que os dois motores construídos ao longo do módulo precisam operar em conjunto: o intertravamento sozinho não enxerga o padrão *bomba ligada + vazão baixa + pressão alta*, e o sistema especialista sozinho não bloqueia o voo por bateria crítica ou vento severo.


In [7]:
DIAGNOSTICOS_CRITICOS_BOMBA = {
    "falta_insumo", "possivel_obstrucao", "possivel_vazamento_ou_falha_bomba", "falha_pulverizacao"
}
DIAGNOSTICOS_CRITICOS_MISSAO = {"interromper_missao"}


def motor_scada_integrado(telemetria: dict) -> dict:
    # --- Derivação dos limiares booleanos (consistentes com a Aula 02) ---
    bat_low = telemetria["bateria"] < 20.0
    wind_high = telemetria["vento"] > 8.0
    tank_empty = telemetria["nivel_reservatorio"] <= 3.0

    # --- 1. Motor de Intertravamento ---
    interlock_decolagem = permissivo_decolagem_drone(
        gps_ok=telemetria["gps_ok"],
        bat_low=bat_low,
        wind_high=wind_high,
        e_stop=telemetria["e_stop"],
        auto_mode=telemetria["auto_mode"],
        manual_mode=telemetria["manual_mode"],
    )
    interlock_pulverizacao = permissivo_pulverizacao(
        is_flying=telemetria["is_flying"],
        alt_ok=telemetria["alt_ok"],
        tank_empty=tank_empty,
    )

    # --- 2. Motor de Diagnóstico ---
    medicoes = {
        "nivel_reservatorio": telemetria["nivel_reservatorio"],
        "vazao": telemetria["vazao"],
        "pressao": telemetria["pressao"],
        "bateria": telemetria["bateria"],
        "vento": telemetria["vento"],
        "bomba_ligada": telemetria["bomba_ligada"],
        "valvula_aberta": telemetria["valvula_aberta"],
    }
    diagnostico = motor_diagnostico(medicoes, regras)
    codigos_ativos = {d["codigo"] for d in diagnostico["diagnosticos"]}

    # --- 3. Fusão: comando final considera intertravamento E diagnóstico ---
    diagnostico_critico_bomba = bool(codigos_ativos & DIAGNOSTICOS_CRITICOS_BOMBA)
    diagnostico_critico_missao = bool(codigos_ativos & DIAGNOSTICOS_CRITICOS_MISSAO)

    comando_liga_bomba = interlock_pulverizacao["Permissivo_Habilitado"] and not diagnostico_critico_bomba
    comando_missao_ativa = interlock_decolagem["Permissivo_Habilitado"] and not diagnostico_critico_missao

    # O intertravamento "achava" que estava tudo liberado, mas o sistema
    # especialista encontrou uma condição de risco não coberta pelos limiares estáticos.
    alerta_apenas_diagnostico_pegou = (
        interlock_pulverizacao["Permissivo_Habilitado"] and diagnostico_critico_bomba
    )

    return {
        "telemetria": telemetria,
        "interlock_decolagem": interlock_decolagem,
        "interlock_pulverizacao": interlock_pulverizacao,
        "diagnostico": diagnostico,
        "comando_liga_bomba": comando_liga_bomba,
        "comando_missao_ativa": comando_missao_ativa,
        "alerta_apenas_diagnostico_pegou": alerta_apenas_diagnostico_pegou,
    }

print("Motor SCADA integrado (Intertravamento + Diagnóstico) pronto.")


Motor SCADA integrado (Intertravamento + Diagnóstico) pronto.


---
## 4. Bateria de Cenários Operacionais

Cada cenário representa uma missão típica (ou uma falha típica) do drone agrícola. O motor integrado é executado para todos eles e o resultado é consolidado em uma tabela para facilitar a análise, como faria a HMI do operador.


In [8]:
cenarios = {
    "Missão ideal": dict(
        gps_ok=True, e_stop=False, auto_mode=True, manual_mode=False,
        is_flying=True, alt_ok=True,
        nivel_reservatorio=80.0, vazao=2.0, pressao=3.0, bateria=75.0, vento=3.0,
        bomba_ligada=True, valvula_aberta=True,
    ),
    "Bateria crítica": dict(
        gps_ok=True, e_stop=False, auto_mode=True, manual_mode=False,
        is_flying=True, alt_ok=True,
        nivel_reservatorio=60.0, vazao=2.0, pressao=3.0, bateria=8.0, vento=3.0,
        bomba_ligada=True, valvula_aberta=True,
    ),
    "Vento severo na decolagem": dict(
        gps_ok=True, e_stop=False, auto_mode=True, manual_mode=False,
        is_flying=False, alt_ok=True,
        nivel_reservatorio=90.0, vazao=0.0, pressao=0.0, bateria=80.0, vento=12.0,
        bomba_ligada=False, valvula_aberta=False,
    ),
    "Possível obstrução dos bicos": dict(
        gps_ok=True, e_stop=False, auto_mode=True, manual_mode=False,
        is_flying=True, alt_ok=True,
        nivel_reservatorio=45.0, vazao=0.25, pressao=5.2, bateria=55.0, vento=4.0,
        bomba_ligada=True, valvula_aberta=True,
    ),
    "Possível vazamento": dict(
        gps_ok=True, e_stop=False, auto_mode=True, manual_mode=False,
        is_flying=True, alt_ok=True,
        nivel_reservatorio=50.0, vazao=0.2, pressao=0.8, bateria=60.0, vento=4.0,
        bomba_ligada=True, valvula_aberta=True,
    ),
    "Reservatório vazio (interlock deveria travar)": dict(
        gps_ok=True, e_stop=False, auto_mode=True, manual_mode=False,
        is_flying=True, alt_ok=True,
        nivel_reservatorio=2.0, vazao=0.0, pressao=0.0, bateria=70.0, vento=3.0,
        bomba_ligada=True, valvula_aberta=True,
    ),
    "Botão de emergência acionado em voo": dict(
        gps_ok=True, e_stop=True, auto_mode=True, manual_mode=False,
        is_flying=True, alt_ok=True,
        nivel_reservatorio=60.0, vazao=2.0, pressao=3.0, bateria=70.0, vento=3.0,
        bomba_ligada=True, valvula_aberta=True,
    ),
}

resumo = []
resultados_completos = {}

for nome, telemetria in cenarios.items():
    resultado = motor_scada_integrado(telemetria)
    resultados_completos[nome] = resultado
    diagnosticos_txt = "; ".join(d["codigo"] for d in resultado["diagnostico"]["diagnosticos"]) or "—"

    resumo.append({
        "Cenário": nome,
        "Permissivo Decolagem": resultado["interlock_decolagem"]["Permissivo_Habilitado"],
        "Permissivo Pulverização (só interlock)": resultado["interlock_pulverizacao"]["Permissivo_Habilitado"],
        "Diagnósticos Ativados": diagnosticos_txt,
        "Comando Final: Liga Bomba": resultado["comando_liga_bomba"],
        "Diagnóstico pegou algo que o interlock não pegou?": resultado["alerta_apenas_diagnostico_pegou"],
    })

df_resumo = pd.DataFrame(resumo)
display(df_resumo)


,Cenário,Permissivo Decolagem,Permissivo Pulverização (só interlock),Diagnósticos Ativados,Comando Final: Liga Bomba,Diagnóstico pegou algo que o interlock não pegou?
0,Missão ideal,True,True,—,True,False
1,Bateria crítica,False,True,interromper_missao,True,False
2,Vento severo na decolagem,False,False,suspender_pulverizacao,False,False
3,Possível obstrução dos bicos,True,True,interromper_pulverizacao; possivel_obstrucao; ...,False,True
4,Possível vazamento,True,True,possivel_vazamento_ou_falha_bomba; falha_pulve...,False,True
5,Reservatório vazio (interlock deveria travar),True,False,falta_insumo; possivel_vazamento_ou_falha_bomb...,False,False
6,Botão de emergência acionado em voo,False,True,—,True,False


Observe a coluna **"Permissivo Pulverização (só interlock)"** comparada com **"Comando Final: Liga Bomba"** nos cenários de *obstrução* e *vazamento*: o intertravamento isolado (que só olha bateria, altitude e tanque vazio) libera a bomba, mas o motor de diagnóstico identifica o padrão de falha e o comando final bloqueia a pulverização. Isso é exatamente a integração que o Módulo 1 se propôs a construir.


---
## 5. Validação Formal do Motor Integrado

Da mesma forma que a Aula 03 provou por exaustão que o estado de perigo `e_stop AND motor_armado` é uma contradição, aqui provamos por exaustão — sobre o espaço de **fatos-base** do sistema especialista — que **sempre que uma condição crítica é diagnosticada, a regra de fusão bloqueia o comando da bomba**. Isso é, na prática, a prova de que a integração dos dois motores preserva a propriedade de segurança validada isoladamente em cada um deles.


In [9]:
fatos_base = ["bomba_ligada", "valvula_aberta", "vazao_baixa", "pressao_alta",
              "pressao_baixa", "nivel_critico", "bateria_critica", "vento_alto"]

violacoes = []
total_estados = 0

for combo in itertools.product([False, True], repeat=len(fatos_base)):
    total_estados += 1
    fatos_iniciais = {fato for fato, ativo in zip(fatos_base, combo) if ativo}

    fatos_finais, _ = forward_chaining(fatos_iniciais, regras)
    codigos_ativos = fatos_finais & DIAGNOSTICOS_CRITICOS_BOMBA

    # Simula a regra de fusão: assumindo que o interlock estático liberou a bomba
    # (pior caso: bateria/altitude/tanque OK), o comando final NUNCA pode ligar
    # a bomba se houver diagnóstico crítico.
    interlock_liberaria = True
    comando_final = interlock_liberaria and not bool(codigos_ativos)

    if codigos_ativos and comando_final:
        violacoes.append(fatos_iniciais)

print(f"Total de combinações de fatos avaliadas: {total_estados}")
print(f"Violações encontradas (diagnóstico crítico com bomba liberada): {len(violacoes)}")

if not violacoes:
    print("CONFIRMADO: em nenhuma combinação de fatos o motor integrado libera a bomba "
          "simultaneamente a um diagnóstico crítico. A propriedade de segurança é preservada pela fusão.")
else:
    print("ALERTA: revisar a regra de fusão, violações encontradas:")
    for v in violacoes:
        print("-", sorted(v))


Total de combinações de fatos avaliadas: 256
Violações encontradas (diagnóstico crítico com bomba liberada): 0
CONFIRMADO: em nenhuma combinação de fatos o motor integrado libera a bomba simultaneamente a um diagnóstico crítico. A propriedade de segurança é preservada pela fusão.


---
## 6. Painel de Apresentação (simulação de HMI)

Por fim, um painel de texto simples ilustra como o resultado do motor integrado poderia ser apresentado ao operador na interface SCADA, reunindo permissivos, diagnósticos e o comando final em um único relatório.


In [10]:
def imprimir_painel_hmi(nome_cenario: str, resultado: dict) -> None:
    print("=" * 60)
    print(f" PAINEL SCADA — {nome_cenario}")
    print("=" * 60)

    print("\n[INTERTRAVAMENTO]")
    print(f"  Permissivo de decolagem : {resultado['interlock_decolagem']['Permissivo_Habilitado']}")
    print(f"  Permissivo de pulverização (estático): {resultado['interlock_pulverizacao']['Permissivo_Habilitado']}")

    print("\n[DIAGNÓSTICO / SISTEMA ESPECIALISTA]")
    diagnosticos = resultado["diagnostico"]["diagnosticos"]
    if diagnosticos:
        for d in diagnosticos:
            print(f"  - [{d['codigo']}] {d['mensagem']}")
    else:
        print("  Nenhuma condição anormal identificada.")

    print("\n[DECISÃO FINAL DO MOTOR INTEGRADO]")
    print(f"  Liga bomba de pulverização? {resultado['comando_liga_bomba']}")
    print(f"  Missão pode prosseguir?     {resultado['comando_missao_ativa']}")
    if resultado["alerta_apenas_diagnostico_pegou"]:
        print("  >> Atenção: apenas o motor de diagnóstico identificou o risco; "
              "o intertravamento estático sozinho teria liberado a bomba.")
    print("=" * 60)


imprimir_painel_hmi("Possível obstrução dos bicos", resultados_completos["Possível obstrução dos bicos"])


 PAINEL SCADA — Possível obstrução dos bicos

[INTERTRAVAMENTO]
  Permissivo de decolagem : True
  Permissivo de pulverização (estático): True

[DIAGNÓSTICO / SISTEMA ESPECIALISTA]
  - [interromper_pulverizacao] Interromper a pulverização.
  - [possivel_obstrucao] Possível obstrução nos bicos ou na tubulação.
  - [falha_pulverizacao] Falha no sistema de pulverização.

[DECISÃO FINAL DO MOTOR INTEGRADO]
  Liga bomba de pulverização? False
  Missão pode prosseguir?     True
  >> Atenção: apenas o motor de diagnóstico identificou o risco; o intertravamento estático sozinho teria liberado a bomba.


---
## Conclusão

O Módulo 1 do projeto SCADA-Core Automática construiu, ao longo de nove aulas, dois motores logicamente independentes: um **motor de intertravamento** baseado em lógica proposicional estática (Aulas 03–05), capaz de provar formalmente a impossibilidade de estados de perigo por limiares de segurança; e um **motor de diagnóstico** baseado em sistemas especialistas (Aulas 07–09), capaz de encadear regras `SE...ENTÃO` para identificar causas raiz de falhas a partir de padrões combinados entre variáveis do processo.

Este notebook demonstrou que a integração dos dois motores é necessária: cenários como *possível obstrução* e *possível vazamento* passam despercebidos por um intertravamento puramente estático, mas são corretamente identificados e bloqueados quando o sistema especialista participa da decisão final. A validação exaustiva da Seção 5 comprova, no mesmo espírito da prova de contradição da Aula 03, que a fusão dos dois motores preserva a propriedade de segurança da planta em qualquer combinação de fatos avaliada.

O motor `motor_scada_integrado` implementado aqui representa a interface que o sistema SCADA do drone agrícola utilizará para transformar telemetria bruta em permissivos, diagnósticos e comandos de atuação seguros, servindo de base para os próximos módulos do projeto (grafos de roteamento, árvores de alarmes e relações de permissão).
